# Fetching Defensive Statistics from FBref

This notebook fetches **per-player, per-match defensive statistics** from FBref for Premier League seasons **2019-20 onwards**.

## Data Structure
Each row in the final dataset represents:
- **One player's defensive stats** in **one match** (which corresponds to one gameweek)

## Defensive Stats Include:
- Tackles (total, won, in defensive/middle/attacking third)
- Pressures (total, successful, in each third)
- Blocks (total, shots blocked, passes blocked)
- Interceptions
- Clearances
- Errors leading to shots

In [2]:
import soccerdata as sd
import pandas as pd
import numpy as np
import time
import warnings

# Suppress the FutureWarning about DataFrame concatenation (it's a pandas/soccerdata internal issue)
warnings.filterwarnings("ignore", category=FutureWarning, module="soccerdata")

# Starting from 2019-20 to 2024-25 (6 seasons with actual defensive data)
seasons = [f"{y}-{str(y+1)[-2:]}" for y in range(2019, 2025)]

print(f"Configured seasons: {seasons}")
print(f"Total: {len(seasons)} seasons")

[01/12/26 00:10:09] INFO     No custom team name replacements found. You can configure these in       ]8;id=325074;file://c:\Users\pc\miniconda3\envs\DM_ENV\Lib\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=702137;file://c:\Users\pc\miniconda3\envs\DM_ENV\Lib\site-packages\soccerdata\_config.py#84\84]8;;\
                             C:\Users\pc\soccerdata\config\teamname_replacements.json.                             

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=240622;file://c:\Users\pc\miniconda3\envs\DM_ENV\Lib\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=700442;file://c:\Users\pc\miniconda3\envs\DM_ENV\Lib\site-packages\soccerdata\_config.py#162\162]8;;\
                             C:\Users\pc\soccerdata\config\league_dict.json.                                       

Configured seasons: ['2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']
Total: 6 seasons


## Step 1: Initialize FBref Scraper and Get Schedule

First, we initialize the FBref scraper and retrieve the match schedule. This gives us all match IDs which we'll use to fetch individual match statistics.

In [2]:
# Initialize the FBref scraper for all selected seasons
fbref = sd.FBref(leagues="ENG-Premier League", seasons=seasons)

# Read the full schedule (contains match IDs, dates, teams, scores)
schedule = fbref.read_schedule()
schedule_df = schedule.reset_index()

# Display schedule structure
print(f"Schedule shape: {schedule_df.shape}")
print(f"Schedule columns: {schedule_df.columns.tolist()[:15]}...")  # First 15 columns
print(f"\nSample schedule rows:")
print(schedule_df.head(3))

                    INFO     Saving cached data to C:\Users\LENOVO\soccerdata\data\FBref             ]8;id=47507;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=358783;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\_common.py#263\263]8;;\

Schedule shape: (2280, 18)
Schedule columns: ['league', 'season', 'game', 'week', 'day', 'date', 'time', 'home_team', 'home_xg', 'score', 'away_xg', 'away_team', 'attendance', 'venue', 'referee']...

Sample schedule rows:
               league season                                  game  week  day  \
0  ENG-Premier League   1920     2019-08-09 Liverpool-Norwich City     1  Fri   
1  ENG-Premier League   1920  2019-08-10 Bournemouth-Sheffield Utd     1  Sat   
2  ENG-Premier League   1920        2019-08-10 Burnley-Southampton     1  Sat   

        date   time    home_team  home_xg score  away_xg      away_team  \
0 2019-08-09  20:00    Liverpool      1.8   4–1      0.9   Norwich City   
1 2019-08-10  15:00  Bournemouth      1.3   1–1      1.3  Sheffield Utd   
2 2019-08-10  15:00      Burnley      0.9   3–0      1.2    Southampton   

   attendance             venue         referee  \
0       53333           Anfield  Michael Oliver   
1       10714  Vitality Stadium    Kevin Friend   

## Step 2: Extract Match IDs and Build Gameweek Mapping

We need to:
1. Extract all unique match IDs from the schedule
2. Create a mapping from match ID to gameweek number and date (for later enrichment)

In [3]:
# Find the match id column (name varies by soccerdata version)
id_candidates = [
    c for c in schedule_df.columns
    if any(k in str(c).lower() for k in ["match_id", "game_id", "matchid", "gameid"])
    or str(c).lower() in {"id", "match"}
    or str(c).lower().endswith("_id")
    or ("match" in str(c).lower() and "id" in str(c).lower())
    or ("game" in str(c).lower() and "id" in str(c).lower())
]

# Find gameweek/round column
gw_candidates = [
    c for c in schedule_df.columns
    if any(k in str(c).lower() for k in ["week", "round", "matchweek", "gameweek", "matchday"])
]

# Find date column
date_candidates = [
    c for c in schedule_df.columns
    if any(k in str(c).lower() for k in ["date", "time", "kickoff"])
]

# Find season column
season_candidates = [
    c for c in schedule_df.columns
    if "season" in str(c).lower()
]

print(f"Match ID candidates: {id_candidates}")
print(f"Gameweek candidates: {gw_candidates}")
print(f"Date candidates: {date_candidates}")
print(f"Season candidates: {season_candidates}")

# Extract match IDs
if id_candidates:
    match_id_col = id_candidates[0]
    match_ids = (
        schedule_df[match_id_col]
        .dropna()
        .astype(str)
        .str.strip()
        .loc[lambda s: s.ne("")]
        .drop_duplicates()
        .tolist()
    )
else:
    # Fallback: try index level names
    idx_match_levels = [
        n for n in schedule.index.names
        if n is not None and any(k in str(n).lower() for k in ["match", "game"]) and "id" in str(n).lower()
    ]
    if not idx_match_levels:
        raise KeyError(
            "Could not identify match id column/level in fbref.read_schedule(). "
            f"Columns: {list(schedule_df.columns)[:30]}; index names: {schedule.index.names}"
        )
    level_name = idx_match_levels[0]
    match_id_col = level_name
    match_ids = (
        pd.Index(schedule.index.get_level_values(level_name))
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
        .tolist()
    )

# Build a mapping DataFrame: match_id -> season, gameweek, date, home_team, away_team
gw_col = gw_candidates[0] if gw_candidates else None
date_col = date_candidates[0] if date_candidates else None
season_col = season_candidates[0] if season_candidates else None

# Build the mapping
mapping_cols = [match_id_col]
if season_col: mapping_cols.append(season_col)
if gw_col: mapping_cols.append(gw_col)
if date_col: mapping_cols.append(date_col)

# Add team columns if available
team_cols = [c for c in schedule_df.columns if "home" in str(c).lower() or "away" in str(c).lower()]
for tc in team_cols[:4]:  # Limit to first 4 team-related columns
    if tc not in mapping_cols:
        mapping_cols.append(tc)

match_mapping = schedule_df[mapping_cols].drop_duplicates()
match_mapping[match_id_col] = match_mapping[match_id_col].astype(str).str.strip()

print(f"\nMatches found in schedule: {len(match_ids):,}")
print(f"Match mapping columns: {match_mapping.columns.tolist()}")
print(f"\nSample mapping:")
print(match_mapping.head(5))

Match ID candidates: ['game_id']
Gameweek candidates: ['week']
Date candidates: ['date', 'time']
Season candidates: ['season']

Matches found in schedule: 2,280
Match mapping columns: ['game_id', 'season', 'week', 'date', 'home_team', 'home_xg', 'away_xg', 'away_team']

Sample mapping:
    game_id season  week       date       home_team  home_xg  away_xg  \
0  928467bd   1920     1 2019-08-09       Liverpool      1.8      0.9   
1  d402cacd   1920     1 2019-08-10     Bournemouth      1.3      1.3   
2  34b99058   1920     1 2019-08-10         Burnley      0.9      1.2   
3  a802f51e   1920     1 2019-08-10  Crystal Palace      0.9      1.1   
4  404ee5d3   1920     1 2019-08-10       Tottenham      2.4      0.7   

       away_team  
0   Norwich City  
1  Sheffield Utd  
2    Southampton  
3        Everton  
4    Aston Villa  


## Step 3: Fetch Defensive Stats for All Matches

This is the main data collection loop. We fetch defensive statistics for each match individually to:
1. Handle failures gracefully (one failed match won't crash the entire process)
2. Track progress and failures for debugging
3. Add gentle pacing to avoid being blocked by FBref

In [ ]:
dfs = []           # Successfully fetched dataframes
failed = []        # Failed match IDs with error info
empty_matches = [] # Matches with no stats available

print(f"Starting to fetch defensive stats for {len(match_ids):,} matches...")
print("This may take a while. Progress updates every 100 matches.\n")

start_time = time.time()

for i, match_id in enumerate(match_ids, start=1):
    try:
        # Fetch defensive stats for this match
        df = fbref.read_player_match_stats(
            stat_type="defense",
            match_id=match_id,
            force_cache=True,  # Use cached data if available
        )
        
        # Check if we got valid data
        if df is not None and not df.empty:
            # Add match_id to the dataframe for later mapping
            df_reset = df.reset_index()
            df_reset['match_id'] = match_id
            dfs.append(df_reset)
        else:
            # Match exists but has no defensive stats (common for older seasons)
            empty_matches.append(match_id)
            
    except Exception as e:
        error_msg = str(e)
        # Only log actual errors, not "no stats" messages
        if "No stats found" not in error_msg:
            failed.append((match_id, type(e).__name__, error_msg[:100]))
        else:
            empty_matches.append(match_id)
    
    # Progress logging every 100 matches
    if i % 100 == 0:
        elapsed = time.time() - start_time
        rate = i / elapsed if elapsed > 0 else 0
        eta = (len(match_ids) - i) / rate if rate > 0 else 0
        print(f"Progress: {i:,}/{len(match_ids):,} matches ({i/len(match_ids)*100:.1f}%)")
        print(f"  ✓ Success: {len(dfs):,} | ⚠ Empty: {len(empty_matches):,} | ✗ Failed: {len(failed):,}")
        print(f"  Time: {elapsed:.0f}s elapsed, ~{eta:.0f}s remaining\n")
        time.sleep(0.3)  # Gentle pacing to avoid rate limiting

# Final summary
total_time = time.time() - start_time
print("="*60)
print("FETCH COMPLETE")
print("="*60)
print(f"Total matches processed: {len(match_ids):,}")
print(f"  ✓ Successfully fetched: {len(dfs):,}")
print(f"  ⚠ Empty (no stats available): {len(empty_matches):,}")
print(f"  ✗ Failed with errors: {len(failed):,}")
print(f"Total time: {total_time/60:.1f} minutes")

In [8]:
# print sample of data to see how to merge with mapping later
if dfs:
    sample_df = dfs[0]
    print("\nSample fetched defensive stats dataframe:")
    print(sample_df.head(3))


Sample fetched defensive stats dataframe:
               league season                               game       team  \
                                                                             
0  ENG-Premier League   1920  2019-08-09 Liverpool-Norwich City  Liverpool   
1  ENG-Premier League   1920  2019-08-09 Liverpool-Norwich City  Liverpool   
2  ENG-Premier League   1920  2019-08-09 Liverpool-Norwich City  Liverpool   

             player jersey_number nation pos     age min  ... Challenges  \
                                                          ...       Lost   
0            Adrián            13    ESP  GK  32-218  52  ...          0   
1           Alisson             1    BRA  GK  26-311  38  ...          0   
2  Andrew Robertson            26    SCO  LB  25-151  90  ...          0   

  Blocks         Int Tkl+Int Clr Err   game_id  match_id  
  Blocks Sh Pass                                          
0      0  0    0   0       0   0   0  928467bd  928467bd  
1      0

In [ ]:
# converting the dfs list into a single dataframe
if dfs:
    defensive_stats_df = pd.concat(dfs, ignore_index=True)
    print(f"\nCombined defensive stats dataframe shape: {defensive_stats_df.shape}")
    # Merging with match mapping to get season, gameweek, date, teams
    


Combined defensive stats dataframe shape: (65788, 28)


In [14]:
defensive_stats_df.to_csv("defensive_stats_raw.csv", index=False)

## Step 4: Transform and Clean Defensive Data for Merging with FPL Dataset

Now we'll transform the raw defensive data to match the format of the `cleaned_merged_seasons.csv` dataset. This includes:
1. Converting season format (1920 → 2019-20)
2. Combining tackle columns (Def 3rd, Mid 3rd, Att 3rd → tackles_total)
3. Standardizing player names to match FPL dataset
4. Extracting gameweek information
5. Selecting only defensive-related columns

In [3]:
# Load the raw defensive data
print("Loading defensive stats raw data...")
df_def = pd.read_csv('defensive_stats_raw.csv', low_memory=False)
print(f"Original defensive data shape: {df_def.shape}")

# Remove the header row (row 0 contains column descriptions)
df_def = df_def[df_def['season'].notna() & (df_def['season'] != '')]
df_def = df_def.reset_index(drop=True)
print(f"After removing header row: {df_def.shape}")

Loading defensive stats raw data...
Original defensive data shape: (65789, 28)
After removing header row: (65788, 28)


In [4]:
# 1. STANDARDIZE SEASON FORMAT (1920 -> 2019-20)
def convert_season_format(season_code):
    """Convert season from '1920' to '2019-20' format"""
    if pd.isna(season_code) or season_code == '':
        return None
    try:
        season_str = str(int(float(season_code)))
        if len(season_str) == 4:
            year1 = int('20' + season_str[:2])
            year2 = season_str[2:]
            return f"{year1}-{year2}"
        return None
    except:
        return None

df_def['season'] = df_def['season'].apply(convert_season_format)
print(f"\nSeasons after conversion:")
print(df_def['season'].value_counts().sort_index())


Seasons after conversion:
season
2019-20    10614
2020-21    10393
2021-22    10485
2022-23    11345
2023-24    11384
2024-25    11567
Name: count, dtype: int64


In [5]:
# 2. COMBINE TACKLE COLUMNS (Def 3rd, Mid 3rd, Att 3rd -> Total Tackles)
print("="*60)
print("COMBINING TACKLE COLUMNS")
print("="*60)

# The columns are named 'Tackles.2' (Def 3rd), 'Tackles.3' (Mid 3rd), 'Tackles.4' (Att 3rd)
# Convert to numeric and combine
tackle_cols = ['Tackles.2', 'Tackles.3', 'Tackles.4']
for col in tackle_cols:
    df_def[col] = pd.to_numeric(df_def[col], errors='coerce')

# Create combined tackles column (sum of all three thirds)
df_def['tackles_total'] = df_def[tackle_cols].sum(axis=1)

print(f"✓ Created 'tackles_total' by combining tackles from all thirds")
print(f"  Sample values: {df_def['tackles_total'].head(10).tolist()}")

COMBINING TACKLE COLUMNS
✓ Created 'tackles_total' by combining tackles from all thirds
  Sample values: [0, 0, 1, 3, 8, 0, 0, 2, 2, 0]


In [6]:
# 2. COMBINE TACKLE COLUMNS (Def 3rd, Mid 3rd, Att 3rd -> Total Tackles)
# The columns are named 'Tackles.2', 'Tackles.3', 'Tackles.4' representing Def 3rd, Mid 3rd, Att 3rd
print("="*60)
print("COMBINING TACKLE COLUMNS")
print("="*60)

# Convert tackle columns to numeric
tackle_cols = ['Tackles.2', 'Tackles.3', 'Tackles.4']  # Def 3rd, Mid 3rd, Att 3rd
for col in tackle_cols:
    df_def[col] = pd.to_numeric(df_def[col], errors='coerce')

# Create combined tackles column (sum of all three thirds)
df_def['tackles_total'] = df_def[tackle_cols].sum(axis=1)

print(f"Created 'tackles_total' column by combining:")
print(f"  - Tackles.2 (Def 3rd)")
print(f"  - Tackles.3 (Mid 3rd)")
print(f"  - Tackles.4 (Att 3rd)")
print(f"\nSample: {df_def['tackles_total'].describe()}")

COMBINING TACKLE COLUMNS
Created 'tackles_total' column by combining:
  - Tackles.2 (Def 3rd)
  - Tackles.3 (Mid 3rd)
  - Tackles.4 (Att 3rd)

Sample: count    65788.000000
mean         1.159832
std          1.421720
min          0.000000
25%          0.000000
50%          1.000000
75%          2.000000
max         11.000000
Name: tackles_total, dtype: float64


In [7]:
# 3. SELECT AND RENAME DEFENSIVE COLUMNS
print("="*60)
print("SELECTING DEFENSIVE COLUMNS")
print("="*60)

# Map raw columns to FPL-style naming
column_mapping = {
    'season': 'season',
    'player': 'name',
    'team': 'team',
    'pos': 'position',
    'min': 'minutes',
    'Tackles': 'tackles',  # Total tackles (Tkl)
    'Tackles.1': 'tackles_won',  # TklW
    'tackles_total': 'tackles_total',  # Combined Def+Mid+Att third
    'Challenges': 'challenges',  # Total challenges
    'Challenges.1': 'challenges_attempted',  # Att
    'Challenges.2': 'challenges_success_rate',  # Tkl%
    'Challenges.3': 'challenges_lost',  # Lost
    'Blocks': 'blocks',  # Total blocks
    'Blocks.1': 'blocks_shots',  # Sh
    'Blocks.2': 'blocks_passes',  # Pass
    'Int': 'interceptions',  # Interceptions
    'Tkl+Int': 'tackles_interceptions',  # Tkl+Int
    'Clr': 'clearances',  # Clearances
    'Err': 'errors',  # Errors leading to shots
    'match_id': 'match_id',
    'game': 'game'
}

# Select only the columns we need for defensive stats
defensive_columns = list(column_mapping.keys())
df_def_selected = df_def[defensive_columns].copy()

# Rename columns to match FPL naming
df_def_selected = df_def_selected.rename(columns=column_mapping)

print(f"Selected columns: {df_def_selected.columns.tolist()}")

SELECTING DEFENSIVE COLUMNS
Selected columns: ['season', 'name', 'team', 'position', 'minutes', 'tackles', 'tackles_won', 'tackles_total', 'challenges', 'challenges_attempted', 'challenges_success_rate', 'challenges_lost', 'blocks', 'blocks_shots', 'blocks_passes', 'interceptions', 'tackles_interceptions', 'clearances', 'errors', 'match_id', 'game']


In [8]:
# 4. CONVERT DATA TYPES AND FILL MISSING VALUES
print("="*60)
print("CONVERTING DATA TYPES")
print("="*60)

# Numeric columns
numeric_cols = [
    'minutes', 'tackles', 'tackles_won', 'tackles_total',
    'challenges', 'challenges_attempted', 'challenges_success_rate', 
    'challenges_lost', 'blocks', 'blocks_shots', 'blocks_passes',
    'interceptions', 'tackles_interceptions', 'clearances', 'errors'
]

for col in numeric_cols:
    df_def_selected[col] = pd.to_numeric(df_def_selected[col], errors='coerce')

# Fill NaN values with 0 for defensive stats (no stat = 0)
df_def_selected[numeric_cols] = df_def_selected[numeric_cols].fillna(0)

print("✓ Converted numeric columns and filled NaN with 0")

CONVERTING DATA TYPES
✓ Converted numeric columns and filled NaN with 0


# here I should verify first if there are nans and then I will see how to fill them not directly nans

In [9]:
# 5. EXTRACT GAMEWEEK FROM MATCH DATA
print("="*60)
print("EXTRACTING GAMEWEEK INFORMATION")
print("="*60)

# Parse the 'game' column to extract date (Format: "2019-08-09 Liverpool-Norwich City")
def extract_date(game_str):
    """Extract date from game string"""
    if pd.isna(game_str) or game_str == '':
        return None
    try:
        parts = str(game_str).split(' ')
        if len(parts) > 0:
            return parts[0]  # Return the date part
    except:
        pass
    return None

df_def_selected['game_date'] = df_def_selected['game'].apply(extract_date)
df_def_selected['game_date'] = pd.to_datetime(df_def_selected['game_date'], errors='coerce')

# Assign gameweek based on season and date
def assign_gameweek(row):
    """Assign gameweek based on season and date"""
    if pd.isna(row['game_date']) or pd.isna(row['season']):
        return None
    
    season = row['season']
    game_date = row['game_date']
    
    try:
        year = int(season.split('-')[0])
    except:
        return None
    
    # Season typically starts in early August
    season_start = pd.Timestamp(f'{year}-08-01')
    
    # Calculate weeks from season start
    weeks_diff = (game_date - season_start).days // 7
    
    # Gameweek is approximately weeks + 1, capped at 38
    gw = min(max(weeks_diff + 1, 1), 38)
    
    return int(gw)

df_def_selected['GW'] = df_def_selected.apply(assign_gameweek, axis=1)

print(f"✓ Assigned gameweeks. GW range: {df_def_selected['GW'].min()} to {df_def_selected['GW'].max()}")

EXTRACTING GAMEWEEK INFORMATION
✓ Assigned gameweeks. GW range: 1 to 38


In [14]:
# 6. CLEAN AND STANDARDIZE PLAYER NAMES TO MATCH FPL DATASET USING FUZZY MATCHING
print("="*60)
print("AUTOMATIC FUZZY NAME MATCHING")
print("="*60)

# Clean player names
df_def_selected['name'] = df_def_selected['name'].str.strip()

# Load FPL dataset to get correct player names
df_fpl = pd.read_csv('data/cleaned_merged_seasons.csv', low_memory=False)
fpl_names = set(df_fpl['name'].dropna().str.strip().unique())
def_names = set(df_def_selected['name'].dropna().unique())

print(f"FPL dataset unique names: {len(fpl_names):,}")
print(f"Defensive dataset unique names: {len(def_names):,}")

# Find names that need matching
names_in_both = fpl_names & def_names
unmatched_def = def_names - fpl_names
available_fpl = fpl_names - def_names

print(f"Names already matching: {len(names_in_both):,}")
print(f"Defensive names to match: {len(unmatched_def):,}")
print(f"Available FPL names: {len(available_fpl):,}")

# Use rapidfuzz for automatic fuzzy matching
from rapidfuzz import fuzz, process

name_mapping = {}
score_threshold = 75  # Minimum similarity score (0-100)

print(f"\nFinding best matches (threshold: {score_threshold})...")
for def_name in sorted(unmatched_def):
    match = process.extractOne(
        def_name, 
        available_fpl, 
        scorer=fuzz.ratio,
        score_cutoff=score_threshold
    )
    if match:
        fpl_name, score, _ = match
        name_mapping[def_name] = fpl_name
        # Remove matched name from available pool to avoid duplicates
        available_fpl.discard(fpl_name)
        print(f"  {def_name:40} -> {fpl_name:40} (score: {score:.1f})")

print(f"\n✓ Generated {len(name_mapping)} automatic name mappings")

# Apply the name mapping
df_def_selected['name'] = df_def_selected['name'].replace(name_mapping)

# Final stats
final_def_names = set(df_def_selected['name'].dropna().unique())
final_matches = len(final_def_names & fpl_names)
print(f"✓ Final matching names: {final_matches:,} / {len(final_def_names):,} ({final_matches/len(final_def_names)*100:.1f}%)")

AUTOMATIC FUZZY NAME MATCHING
FPL dataset unique names: 1,619
Defensive dataset unique names: 1,340
Names already matching: 1,028
Defensive names to match: 312
Available FPL names: 591


ModuleNotFoundError: No module named 'rapidfuzz'

In [ ]:
# 7. FINAL COLUMN SELECTION - KEEP ONLY DEFENSIVE CONTRIBUTION COLUMNS
print("="*60)
print("FINAL COLUMN SELECTION")
print("="*60)

# Select final columns for merging with FPL dataset
final_columns = [
    'season',
    'name',
    'position',
    'team',
    'GW',
    'minutes',
    'tackles',
    'tackles_won',
    'tackles_total',  # Combined tackles from all thirds
    'challenges',
    'challenges_attempted',
    'challenges_success_rate',
    'challenges_lost',
    'blocks',
    'blocks_shots',
    'blocks_passes',
    'interceptions',
    'tackles_interceptions',
    'clearances',
    'errors'
]

df_final = df_def_selected[final_columns].copy()

# Remove rows with missing critical data
df_final = df_final.dropna(subset=['season', 'name', 'GW'])

print(f"✓ Final columns: {len(df_final.columns)}")
print(f"✓ Final shape: {df_final.shape}")
print(f"✓ Unique players: {df_final['name'].nunique():,}")
print(f"✓ Seasons: {sorted(df_final['season'].unique())}")

# Display sample
print("\nSample data:")
print(df_final[['season', 'name', 'GW', 'tackles_total', 'interceptions', 'clearances']].head(10))

FINAL COLUMN SELECTION
✓ Final columns: 20
✓ Final shape: (65788, 20)
✓ Unique players: 1,340
✓ Seasons: ['2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']

Sample data:
    season                            name  GW  tackles_total  interceptions  \
0  2019-20  Adrián San Miguel del Castillo   2              0            0.0   
1  2019-20           Alisson Ramses Becker   2              0            0.0   
2  2019-20                Andrew Robertson   2              1            1.0   
3  2019-20                    Divock Origi   2              3            1.0   
4  2019-20          Fabio Henrique Tavares   2              8            2.0   
5  2019-20             Georginio Wijnaldum   2              0            1.0   
6  2019-20                    James Milner   2              0            0.0   
7  2019-20                       Joe Gomez   2              2            0.0   
8  2019-20                Jordan Henderson   2              2            2.0   
9  2019-20    

In [13]:
# 8. SAVE CLEANED DEFENSIVE DATA
output_file = 'defensive_stats_cleaned.csv'
df_final.to_csv(output_file, index=False)

print("="*60)
print("TRANSFORMATION COMPLETE!")
print("="*60)
print(f"✓ Cleaned defensive data saved to: {output_file}")
print(f"\nData summary:")
print(f"  - Total rows: {len(df_final):,}")
print(f"  - Unique players: {df_final['name'].nunique():,}")
print(f"  - Seasons: {sorted(df_final['season'].unique())}")
print(f"\nRecords per season:")
for season in sorted(df_final['season'].unique()):
    count = len(df_final[df_final['season'] == season])
    print(f"  {season}: {count:,}")

print(f"\nThis file is ready to merge with 'cleaned_merged_seasons.csv'")
print(f"Merge on: ['season', 'name', 'GW']")

TRANSFORMATION COMPLETE!
✓ Cleaned defensive data saved to: defensive_stats_cleaned.csv

Data summary:
  - Total rows: 65,788
  - Unique players: 1,340
  - Seasons: ['2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']

Records per season:
  2019-20: 10,614
  2020-21: 10,393
  2021-22: 10,485
  2022-23: 11,345
  2023-24: 11,384
  2024-25: 11,567

This file is ready to merge with 'cleaned_merged_seasons.csv'
Merge on: ['season', 'name', 'GW']


## Step 5: How to Merge with FPL Dataset

The cleaned defensive data can now be merged with `cleaned_merged_seasons.csv` using the following approach:

```python
# Load both datasets
df_fpl = pd.read_csv('data/cleaned_merged_seasons.csv')
df_defensive = pd.read_csv('defensive_stats_cleaned.csv')

# Rename season column in defensive data to match FPL
df_defensive = df_defensive.rename(columns={'season': 'season_x'})

# Merge on season, player name, and gameweek
merged = df_fpl.merge(
    df_defensive[['season_x', 'name', 'GW', 'tackles', 'tackles_won', 'tackles_total',
                   'challenges', 'interceptions', 'clearances', 'blocks', 'errors']],
    on=['season_x', 'name', 'GW'],
    how='left'  # Keep all FPL records, add defensive stats where available
)
```

**Note**: Not all players will have defensive stats (e.g., players who didn't play that gameweek, or matches without defensive data). The merge will add NaN values for these cases, which you can fill with 0 if needed.